<a href="https://colab.research.google.com/github/jumafernandez/modelado-gestion-datos-untref/blob/main/codigo/python_api_georef.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Acceso a datos con la API Georef Argentina

**Modelado y Gestión de Datos - UNTREF - 2026**

En este notebook consultamos la API abierta de **Georef Argentina**, el servicio de normalización de datos geográficos del Estado nacional, que no requiere credenciales. Vamos a:

- consultar la API y observar la respuesta;
- identificar qué datos devuelve;
- armar una tabla con las provincias;
- modificar la consulta para buscar una provincia en particular;
- ubicar las provincias en un mapa.

Cada celda se ejecuta con **Shift + Enter** o con el botón ▶ de su izquierda.

## 1. ¿Qué es una API en este ejemplo?

Una **API** nos permite pedir datos a un servicio usando una dirección web.

En este caso vamos a consultar esta URL:

<https://apis.datos.gob.ar/georef/api/provincias>

Cuando abrimos esa dirección, el servidor nos devuelve datos estructurados. La respuesta viene en formato **JSON**, que en Python se puede trabajar como una combinación de:

- diccionarios (`dict`),
- listas (`list`),
- textos (`str`),
- números (`int` o `float`).

## 2. Importamos las bibliotecas necesarias

Usaremos:

- `requests` para hacer pedidos a la API;
- `pandas` para organizar los resultados en forma de tabla;
- `json` para ver mejor la estructura de la respuesta.

In [ ]:
import requests
import pandas as pd
import json

## 3. Consultamos todas las provincias

Primero guardamos la URL en una variable y luego hacemos el pedido con `requests.get()`.

La variable `respuesta` contiene lo que devuelve el servidor.

In [ ]:
url = "https://apis.datos.gob.ar/georef/api/provincias"

respuesta = requests.get(url)

print("Código de estado:", respuesta.status_code)

### ¿Qué significa el código de estado?

Si el código de estado es **200**, significa que el pedido se realizó correctamente.

Ahora convertimos la respuesta JSON a una estructura de Python usando `.json()`.

In [ ]:
datos = respuesta.json()

print(type(datos))
print(datos.keys())

In [ ]:
datos

## 4. Observamos la respuesta completa

La respuesta es grande, por eso mostramos solo los primeros caracteres con formato más legible.

In [ ]:
print(json.dumps(datos, indent=2, ensure_ascii=False)[:2000])

## 5. Identificamos qué información devuelve la API

La clave más importante para este ejercicio es `provincias`.

Dentro de `provincias` hay una lista. Cada elemento de esa lista representa una provincia.

In [ ]:
datos.keys()

In [ ]:
provincias = datos["provincias"]

print("Cantidad de provincias encontradas:", len(provincias))
print("Tipo de dato de provincias:", type(provincias))
print("Tipo de dato de la primera provincia:", type(provincias[0]))

provincias[0]

## 6. Armamos una tabla con las provincias

Para trabajar más cómodamente, convertimos la lista de provincias en un `DataFrame`.

Como la coordenada del centroide está anidada dentro de otra estructura, la separamos en dos columnas: `latitud` y `longitud`.

In [ ]:
filas = []

for provincia in provincias:
    fila = {
        "id": provincia["id"],
        "nombre": provincia["nombre"],
        "latitud": provincia["centroide"]["lat"],
        "longitud": provincia["centroide"]["lon"]
    }
    filas.append(fila)

df_provincias = pd.DataFrame(filas)

df_provincias.head()

## 7. Buscamos nombres de provincias dentro de la respuesta

Primero mostramos todos los nombres disponibles.

In [ ]:
for nombre in df_provincias["nombre"]:
    print(nombre)

Ahora buscamos si aparece una provincia determinada.

Podés cambiar el valor de la variable `provincia_buscada`.

In [ ]:
provincia_buscada = "Córdoba"

coincidencias = df_provincias[df_provincias["nombre"].str.lower() == provincia_buscada.lower()]

coincidencias

## 8. Modificamos la consulta para buscar una provincia particular

Para consultar una provincia específica, se agrega un parámetro a la URL:

<https://apis.datos.gob.ar/georef/api/provincias?nombre=cordoba>

En vez de escribir toda la URL manualmente, podemos usar el argumento `params` de `requests.get()`.

Esto suele ser más claro porque los parámetros quedan en un diccionario de Python.

In [ ]:
url_base = "https://apis.datos.gob.ar/georef/api/provincias"

parametros = {
    "nombre": "cordoba"
}

respuesta_cordoba = requests.get(url_base, params=parametros)

print("URL consultada:", respuesta_cordoba.url)
print("Código de estado:", respuesta_cordoba.status_code)

In [ ]:
datos_cordoba = respuesta_cordoba.json()

print(json.dumps(datos_cordoba, indent=2, ensure_ascii=False))

## 9. Pasamos la consulta de una provincia a una tabla

Aunque consultemos una sola provincia, la API igualmente devuelve una lista dentro de la clave `provincias`.

Por eso usamos el mismo procedimiento que antes.

In [ ]:
provincias_cordoba = datos_cordoba["provincias"]

filas = []

for provincia in provincias_cordoba:
    fila = {
        "id": provincia["id"],
        "nombre": provincia["nombre"],
        "latitud": provincia["centroide"]["lat"],
        "longitud": provincia["centroide"]["lon"]
    }
    filas.append(fila)

df_cordoba = pd.DataFrame(filas)

df_cordoba

## 10. Probamos con otra provincia

Cambiá el valor de `nombre_provincia` y ejecutá la celda.

Ejemplos posibles:

- `mendoza`
- `santa fe`
- `buenos aires`
- `chaco`

In [ ]:
nombre_provincia = "mendoza"

parametros = {
    "nombre": nombre_provincia
}

respuesta_provincia = requests.get(url_base, params=parametros)
datos_provincia = respuesta_provincia.json()

print("URL consultada:", respuesta_provincia.url)
print("Cantidad de resultados:", len(datos_provincia["provincias"]))

pd.DataFrame([
    {
        "id": provincia["id"],
        "nombre": provincia["nombre"],
        "latitud": provincia["centroide"]["lat"],
        "longitud": provincia["centroide"]["lon"]
    }
    for provincia in datos_provincia["provincias"]
])

## 11. Las provincias en un mapa

Con los centroides armamos un mapa interactivo con `folium`: cada marcador es una provincia.

Como mapa de fondo usamos el de Esri (`tiles="Esri.WorldStreetMap"`). El de OpenStreetMap, que `folium` usa por defecto, bloquea los pedidos que llegan desde Colab y en su lugar muestra el aviso *Access blocked*.

In [ ]:
import folium

# Obtener el centroide promedio para centrar el mapa en Argentina
lat_centro = df_provincias['latitud'].mean()
lon_centro = df_provincias['longitud'].mean()

# Crear un mapa centrado en Argentina, con el mapa de fondo de Esri
mapa_provincias = folium.Map(location=[lat_centro, lon_centro], zoom_start=4, tiles="Esri.WorldStreetMap")

# Agregar un marcador para cada provincia
for index, row in df_provincias.iterrows():
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        popup=row['nombre'],
        tooltip=row['nombre']
    ).add_to(mapa_provincias)

# Mostrar el mapa
mapa_provincias

## 12. En resumen

### i. ¿Qué datos aparecen asociados a la provincia consultada?

Para cada provincia aparecen datos como:

- `id`: identificador de la provincia;
- `nombre`: nombre de la provincia;
- `categoria`: tipo de unidad territorial, en este caso provincia;
- `centroide`: coordenadas aproximadas del centro geográfico de la provincia, con `lat` y `lon`.

### ii. ¿Qué forma tiene la respuesta obtenida?

La respuesta tiene forma de **JSON**.
En Python la trabajamos como un **diccionario** que contiene varias claves. Una de las claves principales es `provincias`, que contiene una **lista** de provincias.

### iii. ¿Qué diferencia hay entre consultar todas las provincias y consultar una sola?

Cuando consultamos todas las provincias, la lista `provincias` trae muchos elementos.
Cuando agregamos el parámetro `nombre`, la API filtra los resultados y devuelve solo la provincia que coincide con la búsqueda, o una lista más pequeña de coincidencias.

## 13. Para seguir practicando

Modificá el valor de `nombre_provincia` y probá con distintas provincias argentinas.

Luego respondé:

1. ¿La API distingue entre mayúsculas y minúsculas?
2. ¿Qué ocurre si escribís el nombre sin tilde?
3. ¿Qué ocurre si escribís solo una parte del nombre?